# Rolling Window Visualizations
This notebook loads saved outputs from `results/coin=*` and builds Plotly charts for all available coins. Figures are repeated coin by coin so multi-coin runs stay readable.

In [212]:
from pathlib import Path
import re

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

METRICS = ["mse", "mae", "qlike"]

# Set to None to include every available coin in results/coin=*
SELECTED_COINS = None

root = Path.cwd()
if not (root / "results").exists() and (root.parent / "results").exists():
    root = root.parent

results_root = root / "results"
if not results_root.exists():
    raise FileNotFoundError(f"Results directory not found: {results_root}")

all_coin_dirs = sorted(p for p in results_root.glob("coin=*") if p.is_dir())
if not all_coin_dirs:
    raise FileNotFoundError(f"No coin directories found under: {results_root}")

if SELECTED_COINS is None:
    coin_dirs = all_coin_dirs
else:
    selected = {c.upper() for c in SELECTED_COINS}
    coin_dirs = [p for p in all_coin_dirs if p.name.replace("coin=", "").upper() in selected]

if not coin_dirs:
    raise ValueError("No matching coin directories were found for SELECTED_COINS.")

COINS = [p.name.replace("coin=", "") for p in coin_dirs]
print(f"Using results root: {results_root}")
print(f"Coins found ({len(COINS)}): {COINS}")

Using results root: c:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results
Coins found (2): ['BTC', 'ETH']


In [213]:
def map_feature_tag_to_experiment(feature_tag: str) -> str:
    feature_tag = str(feature_tag)
    if feature_tag == "base":
        return "baseline"
    if feature_tag.startswith("base+"):
        parts = [part for part in feature_tag.replace("base+", "").split("+") if part]
        return "with_" + "_and_".join(parts)
    return feature_tag


def extract_model_name(path: Path) -> str:
    match = re.search(r"model=([^_]*)", path.name)
    return match.group(1) if match else "unknown_model"


summary_frames = []
pred_frames = []

for coin_dir in coin_dirs:
    coin = coin_dir.name.replace("coin=", "")
    summary_files = sorted(coin_dir.glob("summary__*.csv"))
    pred_files = sorted(coin_dir.glob("predictions__*.parquet"))

    if not summary_files and not pred_files:
        print(f"Skipping {coin}: no summary or prediction files")
        continue

    for fp in summary_files:
        s = pd.read_csv(fp)
        if "feature_tag" not in s.columns:
            feature_match = re.search(r"features=([^.]*)", fp.name)
            s["feature_tag"] = feature_match.group(1) if feature_match else "unknown"
        if "model_name" not in s.columns:
            s["model_name"] = extract_model_name(fp)
        if "model_kind" not in s.columns:
            s["model_kind"] = s["model_name"].astype(str).str.split("-", n=1).str[0].str.upper()
        s["experiment"] = s["feature_tag"].astype(str).map(map_feature_tag_to_experiment)
        s["coin"] = coin
        s["variant_label"] = s["model_name"].astype(str)
        summary_frames.append(s)

    for fp in pred_files:
        p = pd.read_parquet(fp)
        feature_match = re.search(r"features=([^.]*)", fp.name)
        feature_tag = feature_match.group(1) if feature_match else "unknown"
        p["feature_tag"] = feature_tag
        p["experiment"] = map_feature_tag_to_experiment(feature_tag)
        p["date"] = pd.to_datetime(p["date"], errors="coerce")
        if "model_name" not in p.columns:
            p["model_name"] = extract_model_name(fp)
        if "model_kind" not in p.columns:
            p["model_kind"] = p["model_name"].astype(str).str.split("-", n=1).str[0].str.upper()
        p["coin"] = coin
        p["variant_label"] = p["model_name"].astype(str)
        pred_frames.append(p)

if not summary_frames:
    raise FileNotFoundError("No summary files were loaded for the selected coins.")
if not pred_frames:
    raise FileNotFoundError("No prediction files were loaded for the selected coins.")

summary_df = pd.concat(summary_frames, ignore_index=True)
pred_df = pd.concat(pred_frames, ignore_index=True)
pred_df = pred_df.dropna(subset=["date", "y_true", "y_pred", "coin"]).copy()

print("Loaded summary rows:", len(summary_df))
print("Loaded prediction rows:", len(pred_df))
print("Coins in summary:", sorted(summary_df["coin"].dropna().astype(str).unique().tolist()))
print("Coins in predictions:", sorted(pred_df["coin"].dropna().astype(str).unique().tolist()))
print("Model variants in summary:", sorted(summary_df["variant_label"].dropna().astype(str).unique().tolist()))
summary_df[["coin", "model_kind", "model_name", "feature_tag", "experiment"]].head()

Loaded summary rows: 42
Loaded prediction rows: 50400
Coins in summary: ['BTC', 'ETH']
Coins in predictions: ['BTC', 'ETH']
Model variants in summary: ['HAR-baseline', 'LSTM-hs32', 'LSTM-hs64']


,coin,model_kind,model_name,feature_tag,experiment
0,BTC,HAR,HAR-baseline,base,baseline
1,BTC,HAR,HAR-baseline,base+exceedance_down,with_exceedance_down
2,BTC,HAR,HAR-baseline,base+exceedance_down+lambda_t,with_exceedance_down_and_lambda_t
3,BTC,HAR,HAR-baseline,base+lambda_t,with_lambda_t
4,BTC,HAR,HAR-baseline,base+ratio_event,with_ratio_event


In [214]:
# Graph 1: one plot per metric and per coin; grouped bars = model variants
coins_for_plots = COINS
all_experiments = sorted(
    summary_df["experiment"].dropna().astype(str).unique().tolist(),
    key=lambda value: (0 if value == "baseline" else 1, value.count("_and_"), value),
)

for coin in coins_for_plots:
    coin_summary = summary_df.loc[summary_df["coin"].astype(str) == coin].copy()
    if coin_summary.empty:
        print(f"Skipping {coin}: no summary rows")
        continue

    for metric in METRICS:
        if metric not in coin_summary.columns:
            print(f"Skipping metric '{metric}' for {coin}: column not found")
            continue

        grp = (
            coin_summary.groupby(["experiment", "model_kind", "variant_label"], as_index=False)[metric]
            .mean(numeric_only=True)
        )
        grp["experiment"] = pd.Categorical(grp["experiment"], categories=all_experiments, ordered=True)
        grp = grp.sort_values(["experiment", "model_kind", "variant_label"])

        fig = px.bar(
            grp,
            x="experiment",
            y=metric,
            color="variant_label",
            pattern_shape="model_kind",
            barmode="group",
            title=f"{coin} | {metric.upper()} by experiment and model variant",
            labels={
                "experiment": "Experiment",
                metric: metric.upper(),
                "variant_label": "Model variant",
                "model_kind": "Model family",
            },
        )
        fig.update_layout(template="plotly_white")
        print(f"Displaying {metric.upper()} comparison for {coin}...")
        fig.show()

Displaying MSE comparison for BTC...


Displaying MAE comparison for BTC...


Displaying QLIKE comparison for BTC...


Displaying MSE comparison for ETH...


Displaying MAE comparison for ETH...


Displaying QLIKE comparison for ETH...


In [215]:
# Graph 2: observed RV (black) vs predictions through history, one figure per coin × variant
coins_for_plots = sorted(pred_df["coin"].dropna().astype(str).unique().tolist())

for coin in coins_for_plots:
    coin_pred = pred_df.loc[pred_df["coin"].astype(str) == coin].copy()

    rv_obs = (
        coin_pred.groupby("date", as_index=False)["y_true"]
        .mean()
        .sort_values("date")
    )

    pred_series = (
        coin_pred.groupby(["date", "variant_label", "experiment"], as_index=False)["y_pred"]
        .mean()
        .sort_values("date")
    )
    pred_series["label"] = pred_series["experiment"].astype(str)

    variants = sorted(pred_series["variant_label"].dropna().astype(str).unique().tolist())

    for variant in variants:
        variant_series = pred_series.loc[pred_series["variant_label"].astype(str) == variant]

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=rv_obs["date"],
                y=rv_obs["y_true"],
                mode="lines",
                name="Observed RV",
                line={"color": "black", "width": 3},
            )
        )

        for label, g in variant_series.groupby("label"):
            fig.add_trace(
                go.Scatter(
                    x=g["date"],
                    y=g["y_pred"],
                    mode="lines",
                    name=label,
                    line={"width": 1.5},
                    opacity=0.9,
                )
            )

        fig.update_layout(
            title=f"{coin} | {variant} — Observed RV vs predictions by experiment",
            xaxis_title="Date",
            yaxis_title="RV",
            template="plotly_white",
            legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0.0},
        )
        print(f"Displaying plot for {coin} - {variant}...")
        fig.show()


Displaying plot for BTC - HAR-baseline...


Displaying plot for BTC - LSTM-hs32...


Displaying plot for BTC - LSTM-hs64...


Displaying plot for ETH - HAR-baseline...


Displaying plot for ETH - LSTM-hs32...


Displaying plot for ETH - LSTM-hs64...
